=========================================================
Internship Feedback Sentiment Analysis
Task: Evaluate emotional tone of intern feedback using NLTK
Dataset: Customer Feedback Dataset (Kaggle) - used as proxy
         for intern feedback from surveys/comments/social media
         https://www.kaggle.com/datasets/vishweshsalodkar/customer-feedback-dataset
=========================================================

In [ ]:
# --- STEP 1: Install & import dependencies ---
!pip install kaggle nltk pandas matplotlib seaborn --quiet

In [ ]:
import pandas as pd
import nltk
import matplotlib.pyplot as plt
import seaborn as sns
from nltk.sentiment.vader import SentimentIntensityAnalyzer

In [ ]:
nltk.download('vader_lexicon')

In [ ]:
# --- STEP 2: Download dataset from Kaggle ---
# Option A: Using kagglehub (no API key setup needed in Colab)
import kagglehub
path = kagglehub.dataset_download("vishweshsalodkar/customer-feedback-dataset")
print("Dataset downloaded to:", path)

In [ ]:
import os
csv_file = [f for f in os.listdir(path) if f.endswith('.csv')][0]
df = pd.read_csv(os.path.join(path, csv_file))

In [ ]:
print(df.head())
print(df.columns)

In [ ]:
# --- STEP 3: Clean & prepare data ---
# Rename relevant column to 'feedback_text' (adjust based on actual column name)
text_col = 'Text' if 'Text' in df.columns else df.columns[0]
df = df.rename(columns={text_col: 'feedback_text'})
df = df.dropna(subset=['feedback_text']).reset_index(drop=True)

In [ ]:
# --- STEP 4: Run sentiment analysis with NLTK VADER ---
sia = SentimentIntensityAnalyzer()

In [ ]:
def classify_sentiment(text):
    score = sia.polarity_scores(str(text))['compound']
    if score >= 0.05:
        return 'Positive', score
    elif score <= -0.05:
        return 'Negative', score
    else:
        return 'Neutral', score

In [ ]:
df[['sentiment_label', 'sentiment_score']] = df['feedback_text'].apply(
    lambda x: pd.Series(classify_sentiment(x))
)

In [ ]:
print(df[['feedback_text', 'sentiment_label', 'sentiment_score']].head(10))

In [ ]:
# --- STEP 5: Summary statistics ---
sentiment_counts = df['sentiment_label'].value_counts()
print("\nSentiment Distribution:\n", sentiment_counts)
print("\nPercentage:\n", (sentiment_counts / len(df) * 100).round(2))

In [ ]:
# --- STEP 6: Visualizations ---
plt.figure(figsize=(6, 6))
colors = {'Positive': '#4CAF50', 'Neutral': '#FFC107', 'Negative': '#F44336'}
sentiment_counts.plot(
    kind='pie', autopct='%1.1f%%',
    colors=[colors[label] for label in sentiment_counts.index],
    startangle=90
)
plt.title('Internship Feedback Sentiment Distribution')
plt.ylabel('')
plt.savefig('sentiment_pie_chart.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))
sns.countplot(data=df, x='sentiment_label', hue='sentiment_label',
              order=['Positive', 'Neutral', 'Negative'],
              palette=colors, legend=False)
plt.title('Feedback Count by Sentiment')
plt.xlabel('Sentiment')
plt.ylabel('Count')
plt.savefig('sentiment_bar_chart.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- STEP 7: Export results ---
df.to_csv('internship_feedback_sentiment_results.csv', index=False)
print("\nResults saved to internship_feedback_sentiment_results.csv")